# Personality Experiment for Student AI

最終更新日時: 2026-07-13 13:35:40 +09:00

同じ知識状態の生徒に、異なる個人特徴を持たせて発話が変わるかを確認するノートブックです。

目的は、別のAIや人間評価者が発話だけを見て個人特徴を推定できるかを実験することです。

## 1. Clone and update repository
 
 最初に必ず実行します。GitHubの最新版を取得し、`LLMCommunicationAI` がColab側のファイルに入っているか確認します。
 
 注意: `.ipynb` のセル内容は、`git pull` だけでは開いているColab画面に反映されないことがあります。セルの中身が古い場合は、GitHubからこのノートブックを開き直してください。

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Hiromu-0219/student-ai-test.git"
PROJECT_DIR = Path("/content/student-ai")

if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}

%cd /content/student-ai
!git status --short --branch
!git pull
!git log -1 --oneline
!grep -n "LLMCommunicationAI" src/observer/__init__.py src/observer/trait_classifier.py
print("Update check finished. If the notebook still shows old import code, reopen the latest notebook from GitHub.")

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Create personality profiles

知識状態は同じにして、性格・心理パラメータだけ変えます。

In [ ]:
from copy import deepcopy
from src.state_manager import StateManager
from src.personality_model import build_personality_profile

state_manager = StateManager("data/students")
base_state = state_manager.load_student("S002")

BASE_KNOWLEDGE = {
    "linear_equation": {
        "level": "medium",
        "score": 50,
        "can_solve_ax_plus_b_equals_c": 50,
        "can_transpose_terms": 50,
        "can_divide_by_coefficient": 50,
        "can_handle_negative_numbers": 50,
        "can_handle_fractions": 50,
    }
}

profiles = {
    "P_LOW_CONFIDENCE": {
        "label": "自信低め・質問多め・不安強め",
        "self_efficacy": "very_low",
        "question_tendency": "high",
        "motivation": "medium",
        "big_five": {
            "openness": "medium",
            "conscientiousness": "medium",
            "extraversion": "low",
            "agreeableness": "high",
            "neuroticism": "very_high",
        },
    },
    "P_DILIGENT": {
        "label": "丁寧・粘り強い・協調的",
        "self_efficacy": "medium",
        "question_tendency": "medium",
        "motivation": "very_high",
        "big_five": {
            "openness": "high",
            "conscientiousness": "very_high",
            "extraversion": "medium",
            "agreeableness": "high",
            "neuroticism": "low",
        },
    },
    "P_SHORT_RESERVED": {
        "label": "短文・質問少なめ・そっけない",
        "self_efficacy": "medium",
        "question_tendency": "very_low",
        "motivation": "low",
        "big_five": {
            "openness": "low",
            "conscientiousness": "low",
            "extraversion": "very_low",
            "agreeableness": "low",
            "neuroticism": "medium",
        },
    },
    "P_CONFIDENT_TALKATIVE": {
        "label": "自信あり・発話多め・新しい方法に前向き",
        "self_efficacy": "very_high",
        "question_tendency": "high",
        "motivation": "high",
        "big_five": {
            "openness": "very_high",
            "conscientiousness": "medium",
            "extraversion": "very_high",
            "agreeableness": "medium",
            "neuroticism": "very_low",
        },
    },
}

for profile_id, config in profiles.items():
    state = deepcopy(base_state)
    state["student_id"] = profile_id
    state["name"] = config["label"]
    state["knowledge_state"] = deepcopy(BASE_KNOWLEDGE)
    state["misconceptions"] = ["移項時に符号を変えるか迷うことがある"]
    state["learning_history"] = []
    state["self_efficacy"] = config["self_efficacy"]
    state["question_tendency"] = config["question_tendency"]
    state["motivation"] = config["motivation"]
    state["big_five"] = config["big_five"]
    state_manager.save_student(state)
    print(profile_id, config["label"])
    print(build_personality_profile(state))
    print("-" * 80)

## 4. Generate student utterances with LLM

In [ ]:
from src.student_ai import StudentAISimulator

USE_MOCK_MODEL = False  # False: 実LLMで発話生成 / True: mockで軽く確認
sim = StudentAISimulator(use_mock_model=USE_MOCK_MODEL)

TEACHER_MESSAGE = "2x + 3 = 11 を解いてください。途中で迷ったところがあれば教えてください。"
utterance_rows = []

for profile_id, config in profiles.items():
    result = sim.respond(profile_id, TEACHER_MESSAGE, update_knowledge=False)
    utterance_rows.append({
        "student_id": profile_id,
        "label": config["label"],
        "answer": result["answer"],
    })
    print("#", profile_id, config["label"])
    print(result["answer"])
    print()

## 5. Blind judgment table

下の表を別AIや人間に見せて、どの性格プロファイルか推定させます。

In [ ]:
import pandas as pd

blind_df = pd.DataFrame([
    {"sample_id": f"sample_{i+1}", "utterance": row["answer"]}
    for i, row in enumerate(utterance_rows)
])
answer_key_df = pd.DataFrame(utterance_rows)

display(blind_df)
print("Answer key")
display(answer_key_df)

blind_df.to_csv("data/assessments/personality_blind_samples.csv", index=False)
answer_key_df.to_csv("data/assessments/personality_answer_key.csv", index=False)
print("CSV saved under data/assessments/")

## 6. Communication AI trait classification

伝達AIが生徒発話を読み、プロファイル分類・特徴推定・先生AIへ渡す要約を作ります。

標準では実LLMで分類します。軽く確認したい場合だけ `USE_LLM_COMMUNICATION_AI=False` にしてください。

In [ ]:
import importlib
import src.observer.trait_classifier as trait_classifier

importlib.reload(trait_classifier)
CommunicationAI = trait_classifier.CommunicationAI
LLMCommunicationAI = trait_classifier.LLMCommunicationAI

PROFILE_ANSWER_KEY = {
    "P_LOW_CONFIDENCE": "A",
    "P_DILIGENT": "B",
    "P_SHORT_RESERVED": "C",
    "P_CONFIDENT_TALKATIVE": "D",
}

USE_LLM_COMMUNICATION_AI = True  # True: sim.llmを使ってLLM分類 / False: ルールベース分類

if USE_LLM_COMMUNICATION_AI:
    if USE_MOCK_MODEL:
        raise ValueError("LLM分類を使う場合は、先に USE_MOCK_MODEL=False で生徒AIを作成してください。")
    communication_ai = LLMCommunicationAI(sim.llm)
    classifier_type = "llm"
else:
    communication_ai = CommunicationAI()
    classifier_type = "rule_based"

classified_rows = []

for row in utterance_rows:
    classification = communication_ai.classify_utterance(
        utterance=row["answer"],
        student_id=row["student_id"],
    ).to_dict()
    traits = classification["trait_estimates"]
    expected_profile = PROFILE_ANSWER_KEY[row["student_id"]]
    classified_rows.append({
        "student_id": row["student_id"],
        "label": row["label"],
        "classifier_type": classifier_type,
        "expected_profile": expected_profile,
        "profile_prediction": classification["profile_prediction"],
        "profile_correct": expected_profile == classification["profile_prediction"],
        "self_efficacy": traits["self_efficacy"],
        "question_tendency": traits["question_tendency"],
        "motivation": traits["motivation"],
        "extraversion": traits["extraversion"],
        "conscientiousness": traits["conscientiousness"],
        "neuroticism": traits["neuroticism"],
        "confidence": classification["confidence"],
        "evidence": " / ".join(classification["evidence"]),
        "teacher_summary": classification["teacher_summary"],
        "recommended_teacher_attention": " / ".join(classification["recommended_teacher_attention"]),
        "answer": row["answer"],
    })

communication_df = pd.DataFrame(classified_rows)
profile_accuracy = communication_df["profile_correct"].mean()

print("classifier_type:", classifier_type)
print("profile classification accuracy:", round(profile_accuracy * 100, 1), "%")
display(communication_df)

communication_df.to_csv("data/assessments/communication_ai_trait_classification.csv", index=False)
print("CSV saved: data/assessments/communication_ai_trait_classification.csv")

## 7. Generate strict prompt for another AI judge

personality_blind_samples.csv をもとに、ChatGPTなど別AIへそのまま貼れる評価プロンプトを .txt で作成します。

出力表が崩れないよう、reason以外の列は選択肢だけを書くように制約しています。

In [ ]:
from pathlib import Path

prompt_template_path = Path("data/prompts/personality_judge_prompt_template.txt")
judge_prompt_path = Path("data/assessments/personality_judge_prompt.txt")

sample_lines = []
for row in blind_df.to_dict(orient="records"):
    sample_lines.append(f"## {row['sample_id']}")
    sample_lines.append(row["utterance"])
    sample_lines.append("")

prompt_template = prompt_template_path.read_text(encoding="utf-8")
judge_prompt = prompt_template.replace("{{SAMPLES}}", "\n".join(sample_lines).strip())
judge_prompt_path.write_text(judge_prompt, encoding="utf-8")

print(f"Saved: {judge_prompt_path}")
print("=" * 80)
print(judge_prompt_path.read_text(encoding="utf-8"))